# SentinelFatal2 — E2E Cross-Validation v3
**Notebook:** `09_e2e_cv_v3.ipynb`  
**SSOT:** `docs/new_training_spec.md`  
**Date:** 2026-02-27  
**Config:** A only (cross_entropy, class_weight=3.9, stride=120, patience=15)  

Run all cells from top to bottom. Resume-safe: re-running skips completed folds.

In [7]:
# ─── Cell GPU_DIAG ─────────────────────────────────────────────────────────
import subprocess, os, sys, gc, torch

def sh(cmd, check=True, capture=True):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}")
    return r.stdout.strip() if capture else ""

# GPU info
try:
    gpu_info = sh("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader")
    print(f"[GPU] {gpu_info}")
except Exception:
    print("[GPU] nvidia-smi not available")

# Kill stale GPU processes
try:
    kernel_pid = os.getpid()
    apps_out = sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader", check=False)
    if apps_out:
        for line in apps_out.splitlines():
            parts = line.strip().split(",")
            if parts:
                try:
                    pid = int(parts[0].strip())
                    if pid != kernel_pid:
                        os.kill(pid, 9)
                        print(f"[GPU] Killed stale PID {pid}")
                except Exception:
                    pass
except Exception:
    pass

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        props = torch.cuda.get_device_properties(0)
        total_mb = props.total_memory // 1024 ** 2
        free_mb  = (props.total_memory - torch.cuda.memory_allocated()) // 1024 ** 2
        print(f"[GPU] {props.name}  {free_mb} MB free / {total_mb} MB total")
    except Exception:
        pass

print("[GPU_DIAG] Done")

[GPU] nvidia-smi not available
[GPU_DIAG] Done


In [ ]:
# ─── Cell SEED — Seeds, Constants, Determinism ──────────────────────────────
import os, sys, random, time
import numpy as np
import torch

_SESSION_START = time.time()  # used by session timeout guard in fold loop

# ── Constants ────────────────────────────────────────────────────────────────
REPO_URL    = "https://github.com/ArielShamay/SentinelFatal2.git"
BRANCH      = "master"
REPO_DIR    = "/content/SentinelFatal2"
CONFIG_PATH = f"{REPO_DIR}/config/train_config.yaml"

SEED             = 42
N_FOLDS          = 5
N_BOOTSTRAP      = 10_000
SPEC_CONSTRAINT  = 0.65
AT_CANDIDATES    = [0.30, 0.35, 0.40, 0.45]
N_FEATURES       = 12
PATIENCE         = 15
EMA_BETA         = 0.8
VAL_EVERY_N_EPOCHS = 5
TIMEOUT_SECONDS  = 180   # seconds: if session has < this left, callback raises TimeoutError

# Config A overrides (flat keys — see docs/new_training_spec.md §4.6)
CONFIG_A_OVERRIDES = {
    'loss':         'cross_entropy',  # → cfg['finetune']['loss']
    'patience':     15,               # → cfg['finetune']['patience'] (was 25 in YAML)
    'train_stride': 120,              # → cfg['finetune']['train_stride'] (training DataLoader)
    'val_stride':   120,              # → cfg['finetune']['val_stride'] (compute_recording_auc stride)
    # NOTE: class weights are computed automatically from train CSV (n_neg/n_pos ≈ 3.9)
    # NOTE: 'window_stride' NOT overridden — val DataLoader uses pretrain.window_stride=900
}

# ── Platform detection ───────────────────────────────────────────────────────
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    raise RuntimeError("No GPU detected — change runtime type to GPU in Colab settings.")

# ── Determinism ──────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"[SEED] OK  platform={'Colab' if IS_COLAB else 'local'}  device={DEVICE}  seed={SEED}")

In [ ]:
# ─── Cell REPO — Clone / Pull + pip install ──────────────────────────────────
import subprocess, sys
from pathlib import Path

def sh(cmd, check=True, capture=True):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}")
    return r.stdout.strip() if capture else ""

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    print(f"[REPO] Cloning {BRANCH} from {REPO_URL} ...")
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    print(f"[REPO] Repo exists — fetching + resetting to origin/{BRANCH} ...")
    # Remove stale lock files to prevent 'index.lock' errors
    for lock in repo_dir.glob(".git/*.lock"):
        lock.unlink(missing_ok=True)
    sh(f"git -C {REPO_DIR} fetch origin {BRANCH}")
    sh(f"git -C {REPO_DIR} reset --hard origin/{BRANCH}")

# Add project root to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install requirements
print("[REPO] Installing requirements ...")
sh(f"pip install -q -r {REPO_DIR}/requirements.txt")

commit = sh(f"git -C {REPO_DIR} rev-parse --short HEAD")
print(f"[REPO] Commit: {commit}")
print("[REPO] Done")

In [ ]:
# ─── Cell GPU_PREFLIGHT — CUDA smoke test ────────────────────────────────────
import torch

x = torch.randn(4, 2, 1800, device=DEVICE)
assert x.shape == (4, 2, 1800), f"Unexpected shape: {x.shape}"
del x
torch.cuda.empty_cache()
print("[GPU_PREFLIGHT] OK — CUDA tensor creation at (4,2,1800) passed")

In [ ]:
# ─── Cell MODULE_RELOAD — Force-reload src.* modules ────────────────────────
import sys, importlib, inspect

# Drop cached src.* and scripts.* to guarantee fresh imports after git reset
to_del = [k for k in sys.modules if k.startswith('src.') or k.startswith('scripts.')]
for k in to_del:
    del sys.modules[k]

# Fresh imports
from src.train.finetune    import train as finetune_train, load_pretrained_checkpoint
from src.train.pretrain    import pretrain
from src.train.utils       import compute_recording_auc
from src.model.patchtst    import PatchTST, load_config
from src.model.heads       import ClassificationHead
from scripts.run_e2e_cv_v2 import (
    generate_cv_splits, write_fold_csvs,
    extract_features_for_split, fit_lr_model, predict_lr,
    at_sweep, clinical_threshold, bootstrap_auc_ci,
)

# Verify finetune.py has been patched per docs/new_training_spec.md §3.5
sig = inspect.signature(finetune_train)
required_params = ['val_every_n_epochs', 'resume_from_epoch', 'per_epoch_callback', 'resume_checkpoint']
missing = [p for p in required_params if p not in sig.parameters]
if missing:
    raise AssertionError(
        f"[MODULE_RELOAD] finetune.py missing params: {missing}\n"
        f"  Apply Section 3.5 changes from docs/new_training_spec.md first."
    )

# Verify original params still present
for p in ['config_overrides', 'save_epoch_ckpts', 'quiet']:
    assert p in sig.parameters, f"Missing original param: {p}"

print(f"[MODULE_RELOAD] finetune.py signature OK — {len(sig.parameters)} params total")
print("[MODULE_RELOAD] All imports OK")

In [ ]:
# ─── Cell DATA_DOWN — Data download + directory setup ───────────────────────
import subprocess, pandas as pd
from pathlib import Path

PROCESSED_ROOT   = Path(REPO_DIR) / "data" / "processed"
CTU_UHB_DIR      = PROCESSED_ROOT / "ctu_uhb"
FHRMA_DIR        = PROCESSED_ROOT / "fhrma"
SPLITS_DIR       = Path(REPO_DIR) / "data" / "splits"
BASE_CKPT        = Path(REPO_DIR) / "checkpoints" / "e2e_cv_v2"
BASE_LOGS        = Path(REPO_DIR) / "logs" / "e2e_cv_v2"
BASE_RESULTS     = Path(REPO_DIR) / "results" / "e2e_cv_v2"
SHARED_PRETRAIN_CKPT_DIR = BASE_CKPT / "shared_pretrain"

# Count existing .npy files (exclude .gitkeep placeholder)
ctu_npys = [f for f in CTU_UHB_DIR.glob("*.npy") if f.name != ".gitkeep"]
print(f"[DATA_DOWN] CTU-UHB .npy count: {len(ctu_npys)}")

if len(ctu_npys) < 552:
    print("[DATA_DOWN] Fewer than 552 files found — downloading data_processed.zip ...")
    zip_url  = "https://raw.githubusercontent.com/ArielShamay/SentinelFatal2/master/data_processed.zip"
    zip_path = "/tmp/data_processed.zip"
    subprocess.run(f"wget -q -O {zip_path} '{zip_url}'", shell=True, check=True)
    subprocess.run(f"unzip -q -o {zip_path} -d {REPO_DIR}/data/", shell=True, check=True)
    subprocess.run(f"rm -f {zip_path}", shell=True)
    ctu_npys = [f for f in CTU_UHB_DIR.glob("*.npy") if f.name != ".gitkeep"]
    print(f"[DATA_DOWN] After download: {len(ctu_npys)} CTU-UHB .npy files")

# Build train_val_test.csv (idempotent)
train_df = pd.read_csv(SPLITS_DIR / "train.csv", dtype={"id": str, "target": int})
val_df   = pd.read_csv(SPLITS_DIR / "val.csv",   dtype={"id": str, "target": int})
test_df  = pd.read_csv(SPLITS_DIR / "test.csv",  dtype={"id": str, "target": int})
all_df   = pd.concat([train_df, val_df, test_df]).drop_duplicates(subset="id").reset_index(drop=True)
assert len(all_df) == 552, f"Expected 552 rows, got {len(all_df)}"
all_df.to_csv(SPLITS_DIR / "train_val_test.csv", index=False)
print(f"[DATA_DOWN] Built train_val_test.csv: {len(all_df)} recordings ({int(all_df.target.sum())} pos)")

# Create all output directories
for d in [
    BASE_LOGS, BASE_RESULTS, SHARED_PRETRAIN_CKPT_DIR,
    *[BASE_CKPT / f"fold{k}" / "finetune" for k in range(N_FOLDS)],
    *[BASE_RESULTS / f"fold{k}_splits" for k in range(N_FOLDS)],
]:
    d.mkdir(parents=True, exist_ok=True)

print("[DATA_DOWN] All directories created — ready")

In [ ]:
# ─── Cell SANITY — Full data integrity check ─────────────────────────────────
import pandas as pd
from pathlib import Path

errors = []

# 1. File counts
ctu_npys = [f for f in CTU_UHB_DIR.glob("*.npy") if f.name != ".gitkeep"]
fhr_npys = [f for f in FHRMA_DIR.glob("*.npy")   if f.name != ".gitkeep"]

for label, got, expected in [
    ("CTU-UHB npy", len(ctu_npys), 552),
    ("FHRMA npy",   len(fhr_npys), 135),
]:
    ok = got >= expected
    print(f"  {'OK' if ok else 'FAIL'} {label}: {got}/{expected}")
    if not ok:
        errors.append(f"{label}: got {got}, need {expected}")

# 2. Split CSVs
for name, exp_rows in [("pretrain.csv", 687), ("train.csv", 441), ("val.csv", 56), ("test.csv", 55)]:
    p = SPLITS_DIR / name
    if not p.exists():
        errors.append(f"Missing splits/{name}")
        print(f"  FAIL splits/{name}: MISSING")
        continue
    df = pd.read_csv(p)
    ok = len(df) >= exp_rows
    print(f"  {'OK' if ok else 'FAIL'} splits/{name}: {len(df)}/{exp_rows} rows")
    if not ok:
        errors.append(f"splits/{name}: got {len(df)}, expected >={exp_rows}")

# 3. Label distribution in train_val_test.csv
all_df  = pd.read_csv(SPLITS_DIR / "train_val_test.csv", dtype={"id": str, "target": int})
n_pos   = int((all_df["target"] == 1).sum())
n_neg   = int((all_df["target"] == 0).sum())
ok_pos  = n_pos == 113
print(f"  {'OK' if ok_pos else 'FAIL'} Label distribution: {n_pos} pos / {n_neg} neg (expect 113/439)")
if not ok_pos:
    errors.append(f"Wrong label distribution: {n_pos} pos (expect 113)")

# 4. All IDs have corresponding .npy files
ctu_id_set   = {f.stem for f in ctu_npys}
missing_npy  = [row.id for _, row in all_df.iterrows() if row.id not in ctu_id_set]
ok_npy       = len(missing_npy) == 0
print(f"  {'OK' if ok_npy else 'FAIL'} All IDs have .npy files (missing: {len(missing_npy)})")
if missing_npy:
    errors.append(f"Missing .npy for: {missing_npy[:5]} ...")

if errors:
    raise AssertionError("[SANITY] FAILED — fix before continuing:\n  " + "\n  ".join(errors))
print("\n[SANITY] ALL CHECKS PASSED")

In [ ]:
# ─── Cell PRETRAIN — Shared pre-training (skip if checkpoint exists) ─────────
import gc, torch
from pathlib import Path

# Look for existing shared pretrain checkpoint (primary and fallback paths)
_ckpt_candidates = [
    SHARED_PRETRAIN_CKPT_DIR / "best_pretrain.pt",
    Path(REPO_DIR) / "checkpoints" / "pretrain" / "best_pretrain.pt",
]

SHARED_PRETRAIN_CKPT = None
for p in _ckpt_candidates:
    if p.exists():
        SHARED_PRETRAIN_CKPT = str(p)
        print(f"[PRETRAIN] Found existing checkpoint: {p}")
        print("[PRETRAIN] Skipping pre-training.")
        break

if SHARED_PRETRAIN_CKPT is None:
    print("[PRETRAIN] No checkpoint found — starting pre-training (this takes ~1-2 h on T4) ...")
    pretrain(
        config_path    = CONFIG_PATH,
        device_str     = DEVICE,
        max_batches    = 0,
        processed_root = str(PROCESSED_ROOT),
        pretrain_csv   = str(SPLITS_DIR / "pretrain.csv"),
        checkpoint_dir = str(SHARED_PRETRAIN_CKPT_DIR),
        log_path       = str(BASE_LOGS / "shared_pretrain_loss.csv"),
    )
    SHARED_PRETRAIN_CKPT = str(SHARED_PRETRAIN_CKPT_DIR / "best_pretrain.pt")
    print(f"[PRETRAIN] Done → {SHARED_PRETRAIN_CKPT}")

gc.collect()
torch.cuda.empty_cache()
print(f"[PRETRAIN] SHARED_PRETRAIN_CKPT = {SHARED_PRETRAIN_CKPT}")

In [ ]:
# ─── Cell G1_GATE — Pre-train quality gate (val_mse < 0.015) ─────────────────
import pandas as pd
from pathlib import Path

_log_candidates = [
    BASE_LOGS / "shared_pretrain_loss.csv",
    Path(REPO_DIR) / "logs" / "pretrain_loss.csv",
]

pretrain_log = next((p for p in _log_candidates if p.exists()), None)

if pretrain_log is None:
    print("[G1_GATE] WARNING: No pre-training log found — cannot verify G1. Proceeding.")
else:
    df_log = pd.read_csv(pretrain_log)
    val_col = next(
        (c for c in df_log.columns if 'val' in c.lower() and ('loss' in c.lower() or 'mse' in c.lower())),
        None
    )
    if val_col:
        min_val = df_log[val_col].min()
        gate_ok = min_val < 0.015
        print(f"[G1_GATE] min_{val_col} = {min_val:.6f}  threshold = 0.015  → {'PASS' if gate_ok else 'FAIL — check pre-training logs'}")
        if not gate_ok:
            print("[G1_GATE] WARNING: Backbone may not be well-converged. Consider re-running pre-training.")
            print("[G1_GATE] Proceeding anyway — operator decision required.")
    else:
        print(f"[G1_GATE] Columns: {list(df_log.columns)} — cannot verify. Proceeding.")

In [ ]:
# ─── Cell CV_SPLITS — Generate 5-fold stratified splits ──────────────────────
import pandas as pd
from pathlib import Path

all_csv   = SPLITS_DIR / "train_val_test.csv"
cv_splits = generate_cv_splits(all_csv, n_folds=N_FOLDS, seed=SEED)

# Print and save split CSVs for each fold
for k, split in enumerate(cv_splits):
    df_all  = split["df_all"]
    n_tr    = len(split["train_ids"])
    n_vl    = len(split["val_ids"])
    n_te    = len(split["test_ids"])
    tr_pos  = int(df_all[df_all["id"].isin(split["train_ids"])]["target"].sum())
    vl_pos  = int(df_all[df_all["id"].isin(split["val_ids"])]["target"].sum())
    te_pos  = int(df_all[df_all["id"].isin(split["test_ids"])]["target"].sum())
    print(f"  Fold {k}: train={n_tr}({tr_pos}+)  val={n_vl}({vl_pos}+)  test={n_te}({te_pos}+)")

    fold_split_dir = BASE_RESULTS / f"fold{k}_splits"
    fold_train_csv, fold_val_csv, fold_test_csv = write_fold_csvs(split, fold_split_dir)
    split["train_csv"] = fold_train_csv
    split["val_csv"]   = fold_val_csv
    split["test_csv"]  = fold_test_csv

print(f"\n[CV_SPLITS] Generated {N_FOLDS} stratified folds from {all_csv.name}")

In [ ]:
# ─── Cell FOLD_LOOP — Main 5-fold training + evaluation ─────────────────────
#
# ⚠️  RUNTIME RECYCLE WARNING
# If the Colab VM was recycled (new VM assigned), the /content/ filesystem is WIPED.
# epoch_*.pt on local disk are GONE. In-progress folds restart from epoch 0.
# Completed fold OOF CSVs are safe (committed to GitHub via mini-commit).
# To minimize lost work: keep the Colab tab open; avoid 'Disconnect and delete runtime'.
#
import gc, glob, os, re, traceback, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score

# ── Helper: shell command ─────────────────────────────────────────────────────
def sh(cmd, check=False):
    import subprocess
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(f"  [sh] WARN: {r.stderr[:200]}")
    return r.stdout.strip()

# ── Helper: prune old epoch checkpoints (keep last `keep` + best_finetune.pt) ─
def _prune_old_checkpoints(ckpt_dir, keep=3):
    pts = sorted(glob.glob(str(Path(ckpt_dir) / "epoch_*.pt")))
    for old in pts[:-keep]:
        try:
            os.remove(old)
        except Exception:
            pass

# ── Helper: detect resume epoch from existing epoch_*.pt files ───────────────
def _detect_resume_epoch(ckpt_dir):
    pts = sorted(glob.glob(str(Path(ckpt_dir) / "epoch_*.pt")))
    if not pts:
        return 0, None
    latest = pts[-1]
    m = re.search(r'epoch_(\d+)\.pt', latest)
    epoch_num = int(m.group(1)) + 1 if m else 0
    return epoch_num, latest

# ── Helper: estimated Colab session time remaining ───────────────────────────
def _session_time_remaining():
    elapsed = time.time() - _SESSION_START
    return max(0.0, 14400 - elapsed)  # Colab T4 free tier ≈ 4 h = 14400 s

# ── Helper: load best finetune model for inference ────────────────────────────
def _load_best_model(ckpt_path, device):
    cfg   = load_config(CONFIG_PATH)
    model = PatchTST(cfg)
    d_in  = int(cfg["data"]["n_patches"]) * int(cfg["model"]["d_model"]) * int(cfg["data"]["n_channels"])
    model.replace_head(ClassificationHead(
        d_in=d_in,
        n_classes=int(cfg["finetune"]["n_classes"]),
        dropout=float(cfg["model"]["dropout"]),
    ))
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model = model.to(device)
    model.eval()
    return model

# ── Helper: post-fold mini-commit to GitHub ───────────────────────────────────
def _try_git_push_oof(fold_k, oof_csv_path):
    try:
        from google.colab import userdata as _ud
        token = _ud.get('GITHUB_TOKEN')
        if token:
            sh(f'git -C {REPO_DIR} remote set-url origin '
               f'"https://{token}@github.com/ArielShamay/SentinelFatal2.git"')
    except Exception:
        pass
    sh(f'git -C {REPO_DIR} add "{oof_csv_path}"')
    sh(f'git -C {REPO_DIR} commit -m "[auto] fold {fold_k} OOF scores"')
    sh(f'git -C {REPO_DIR} push origin master')
    sh(f'git -C {REPO_DIR} remote set-url origin "https://github.com/ArielShamay/SentinelFatal2.git"')
    print(f"[FOLD {fold_k}] OOF CSV committed to GitHub")

# ── DataLoader monkey-patch: force num_workers=0 (spec Rule 3) ───────────────
# In finetune.py, num_workers is hardcoded as min(2, os.cpu_count()).  On Colab
# T4 (multi-core) this evaluates to 2.  Combined with SWA this caused the
# epoch-66 deadlock.  Patching DataLoader globally before every finetune_train()
# call is the safest fix.
import torch.utils.data as _tud
_orig_DataLoader = _tud.DataLoader

def _patched_DataLoader(*args, **kwargs):
    kwargs['num_workers'] = 0
    kwargs['pin_memory']  = False
    return _orig_DataLoader(*args, **kwargs)

_tud.DataLoader = _patched_DataLoader

# ── FOLD LOOP ─────────────────────────────────────────────────────────────────
oof_results = {}
_timed_out  = False

for k in range(N_FOLDS):
    fold_ckpt_dir  = str(BASE_CKPT / f"fold{k}" / "finetune")
    fold_train_csv = cv_splits[k]["train_csv"]
    fold_val_csv   = cv_splits[k]["val_csv"]
    fold_test_csv  = cv_splits[k]["test_csv"]
    oof_csv_path   = BASE_RESULTS / f"fold{k}_oof_scores.csv"

    print(f"\n{'═'*54}")
    print(f" FOLD {k}/{N_FOLDS}  ({_session_time_remaining():.0f}s remaining in session)")
    print(f"{'═'*54}")

    # Skip if fold already complete (OOF CSV exists)
    if oof_csv_path.exists():
        print(f"[FOLD {k}] OOF CSV found — skipping (already complete)")
        oof_results[k] = pd.read_csv(oof_csv_path)
        continue

    # Guard: stop before starting a new fold if < 30 min remain
    if _session_time_remaining() < 1800:
        print(f"[FOLD {k}] Less than 30 min remaining — aborting to preserve prior results")
        _timed_out = True
        break

    # Detect resume epoch from existing epoch_*.pt files
    resume_epoch, resume_ckpt_path = _detect_resume_epoch(fold_ckpt_dir)
    if resume_epoch > 0:
        print(f"[FOLD {k}] Resuming from epoch {resume_epoch} (checkpoint: {resume_ckpt_path})")
    else:
        print(f"[FOLD {k}] Starting fresh from epoch 0")

    # Per-epoch callback: prune checkpoints + memory flush + timeout guard
    # Also tracks the epoch with the highest observed (raw) val_auc for the summary (spec §7.k)
    _best_epoch_tracker = {'epoch': -1, 'val_auc': 0.0}

    def _epoch_callback(epoch, train_loss, val_auc, _k=k, _ckpt_dir=fold_ckpt_dir,
                        _tracker=_best_epoch_tracker):
        _prune_old_checkpoints(_ckpt_dir, keep=3)
        gc.collect()
        torch.cuda.empty_cache()
        # Track approximate best epoch (raw val_auc; finetune.py saves best_finetune.pt on EMA)
        if val_auc > _tracker['val_auc']:
            _tracker['val_auc'] = val_auc
            _tracker['epoch']   = epoch
        remaining = _session_time_remaining()
        if remaining < TIMEOUT_SECONDS:
            print(f"\n[TIMEOUT GUARD] Fold {_k} epoch {epoch} — {remaining:.0f}s remaining — raising TimeoutError")
            raise TimeoutError(f"Session timeout guard triggered at epoch {epoch}")

    # ── Finetune training ─────────────────────────────────────────────────────
    try:
        finetune_train(
            config_path         = CONFIG_PATH,
            device_str          = DEVICE,
            max_batches         = 0,
            processed_root      = str(PROCESSED_ROOT),
            train_csv           = str(fold_train_csv),
            val_csv             = str(fold_val_csv),
            pretrain_checkpoint = str(SHARED_PRETRAIN_CKPT),
            checkpoint_dir      = fold_ckpt_dir,
            log_path            = str(BASE_LOGS / f"fold{k}_finetune_loss.csv"),
            quiet               = True,
            save_epoch_ckpts    = True,
            config_overrides    = CONFIG_A_OVERRIDES,
            val_every_n_epochs  = VAL_EVERY_N_EPOCHS,   # AUC computed every 5 epochs
            resume_from_epoch   = resume_epoch,
            resume_checkpoint   = resume_ckpt_path,     # load model weights from latest epoch_*.pt
            per_epoch_callback  = _epoch_callback,
        )
    except TimeoutError as e:
        print(f"[FOLD {k}] TimeoutError: {e} — stopping fold loop")
        _timed_out = True
        break
    except KeyboardInterrupt:
        print(f"[FOLD {k}] KeyboardInterrupt — stopping fold loop")
        _timed_out = True
        break
    except Exception:
        print(f"[FOLD {k}] ERROR during finetune_train — skipping this fold")
        traceback.print_exc()
        continue
    finally:
        # Re-apply patch defensively — keeps num_workers=0 for next fold iteration
        # (spec §7.d: patch stays active across all folds; _orig_DataLoader restored after loop)
        _tud.DataLoader = _patched_DataLoader

    # ── Load best model ───────────────────────────────────────────────────────
    best_ckpt = Path(fold_ckpt_dir) / "best_finetune.pt"
    if not best_ckpt.exists():
        print(f"[FOLD {k}] best_finetune.pt not found — skipping evaluation")
        continue
    try:
        model = _load_best_model(str(best_ckpt), DEVICE)
        print(f"[FOLD {k}] Loaded best model from {best_ckpt.name}")
    except Exception:
        print(f"[FOLD {k}] ERROR loading model — skipping evaluation")
        traceback.print_exc()
        continue

    # ── AT Sweep ──────────────────────────────────────────────────────────────
    try:
        best_at, best_val_auc, at_results = at_sweep(
            model, fold_val_csv, PROCESSED_ROOT,
            train_csv=fold_train_csv, device=DEVICE,
            inference_stride=24, n_features=N_FEATURES,
            lr_C=0.1, use_pca=True,
        )
        print(f"[FOLD {k}] AT sweep → best_at={best_at:.2f}  val_auc={best_val_auc:.4f}")
        at_df = pd.DataFrame([{"at": at, "val_auc": v, "fold": k} for at, v in at_results.items()])
        at_df.to_csv(BASE_RESULTS / f"fold{k}_at_sweep.csv", index=False)
    except Exception:
        print(f"[FOLD {k}] AT sweep failed — defaulting to AT=0.40")
        traceback.print_exc()
        best_at, best_val_auc = 0.40, 0.0

    # ── Feature extraction ────────────────────────────────────────────────────
    try:
        X_tr, y_tr, _      = extract_features_for_split(model, fold_train_csv, PROCESSED_ROOT, best_at, 24, DEVICE, N_FEATURES)
        X_vl, y_vl, _      = extract_features_for_split(model, fold_val_csv,   PROCESSED_ROOT, best_at, 24, DEVICE, N_FEATURES)
        X_te, y_te, te_ids = extract_features_for_split(model, fold_test_csv,  PROCESSED_ROOT, best_at, 24, DEVICE, N_FEATURES)
    except Exception:
        print(f"[FOLD {k}] Feature extraction failed — skipping")
        traceback.print_exc()
        del model
        gc.collect(); torch.cuda.empty_cache()
        continue

    # ── LR model + clinical threshold ─────────────────────────────────────────
    X_tv = np.concatenate([X_tr, X_vl])
    y_tv = np.concatenate([y_tr, y_vl])
    scaler, pca, lr_m   = fit_lr_model(X_tv, y_tv, C=0.1, use_pca=True)
    test_scores         = predict_lr(X_te, scaler, pca, lr_m)
    val_scores          = predict_lr(X_vl, scaler, pca, lr_m)
    test_auc            = roc_auc_score(y_te, test_scores)
    # Threshold selected on val (NOT on test — avoid leakage)
    threshold_primary, sensitivity, specificity = clinical_threshold(y_vl, val_scores, SPEC_CONSTRAINT)
    y_pred_te           = (test_scores >= threshold_primary).astype(int)

    # ── Save OOF CSV ──────────────────────────────────────────────────────────
    oof_df = pd.DataFrame({
        "id":                te_ids,
        "y_true":            y_te.tolist(),
        "y_score":           test_scores.tolist(),
        "y_pred":            y_pred_te.tolist(),
        "best_at":           best_at,
        "threshold_primary": threshold_primary,
        "fold":              k,
    })
    oof_df.to_csv(oof_csv_path, index=False)
    print(f"[FOLD {k}] OOF CSV saved ({len(oof_df)} recordings)")
    oof_results[k] = oof_df

    # Mini-commit OOF CSV to GitHub (survives runtime recycle)
    _try_git_push_oof(k, str(oof_csv_path))

    # ── G3 Gate (after Fold 0 only) ───────────────────────────────────────────
    if k == 0 and test_auc < 0.65:
        print(f"\n[G3 GATE] FAIL — fold0 test AUC = {test_auc:.4f} < 0.65")
        print("[G3 GATE] Possible causes: class weights not applied, pretrain ckpt wrong, stride issue")
        input("[G3 GATE] Press Enter to continue to Fold 1, or Ctrl+C to abort: ")

    # ── Per-fold summary (spec §7.k) ──────────────────────────────────────────
    _best_ep = _best_epoch_tracker['epoch']
    print(f"\n{'═'*54}")
    print(f" FOLD {k} COMPLETE")
    print(f"   Test AUC:    {test_auc:.4f}")
    print(f"   Sensitivity: {sensitivity:.4f}  (val-threshold, spec≥{SPEC_CONSTRAINT})")
    print(f"   Specificity: {specificity:.4f}")
    print(f"   Best AT:     {best_at:.2f}")
    print(f"   Threshold:   {threshold_primary:.4f}")
    print(f"   Best Epoch:  {_best_ep}  (approx. highest raw val_auc={_best_epoch_tracker['val_auc']:.4f})")
    print(f"{'═'*54}")

    # Free GPU memory before next fold
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Restore original DataLoader after fold loop
_tud.DataLoader = _orig_DataLoader

if _timed_out:
    print("\n[FOLD_LOOP] Session timeout — partial results saved. Re-run notebook to resume.")
else:
    print(f"\n[FOLD_LOOP] All {N_FOLDS} folds complete")


In [ ]:
# ─── Cell AT_SUMMARY — AT sweep results across all folds ─────────────────────
import pandas as pd
from pathlib import Path

rows = []
for k in range(N_FOLDS):
    p = BASE_RESULTS / f"fold{k}_at_sweep.csv"
    if p.exists():
        df = pd.read_csv(p)
        for _, row in df.iterrows():
            rows.append({"fold": k, "at": row.get("at", "?"), "val_auc": row.get("val_auc", 0.0)})

if rows:
    summary = pd.DataFrame(rows)
    try:
        pivot = summary.pivot(index="fold", columns="at", values="val_auc")
        print("[AT_SUMMARY] Val AUC per fold × AT threshold:")
        print(pivot.round(4).to_string())
    except Exception:
        print(summary.to_string(index=False))
    summary.to_csv(BASE_RESULTS / "at_sweep_summary.csv", index=False)
    print("[AT_SUMMARY] Saved at_sweep_summary.csv")
else:
    print("[AT_SUMMARY] No AT sweep results yet — run FOLD_LOOP first")

In [ ]:
# ─── Cell BOOTSTRAP_CI — Global OOF AUC + 95% confidence interval ────────────
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

oof_dfs    = []
fold_aucs  = []
fold_rows  = []

for k in range(N_FOLDS):
    p = BASE_RESULTS / f"fold{k}_oof_scores.csv"
    if not p.exists():
        print(f"[BOOTSTRAP_CI] Fold {k} OOF missing — skipping")
        continue
    df = pd.read_csv(p, dtype={"id": str})
    oof_dfs.append(df)
    fauc = roc_auc_score(df["y_true"], df["y_score"])
    fold_aucs.append(fauc)
    t_k  = df["threshold_primary"].iloc[0]
    at_k = df["best_at"].iloc[0]
    sens_k  = float((df.loc[df["y_true"] == 1, "y_pred"] == 1).mean()) if (df["y_true"] == 1).sum() > 0 else 0.0
    spec_k  = float((df.loc[df["y_true"] == 0, "y_pred"] == 0).mean()) if (df["y_true"] == 0).sum() > 0 else 0.0
    fold_rows.append({"fold": k, "auc": fauc, "sensitivity": sens_k, "specificity": spec_k,
                      "best_at": at_k, "threshold": t_k, "n": len(df), "n_pos": int(df["y_true"].sum())})
    print(f"  Fold {k}: AUC={fauc:.4f}  sens={sens_k:.4f}  spec={spec_k:.4f}  AT={at_k:.2f}")

if not oof_dfs:
    print("[BOOTSTRAP_CI] No OOF data available — run FOLD_LOOP first")
else:
    global_oof = pd.concat(oof_dfs).reset_index(drop=True)
    global_oof.to_csv(BASE_RESULTS / "global_oof_predictions.csv", index=False)

    y_true  = global_oof["y_true"].values
    y_score = global_oof["y_score"].values

    global_auc = roc_auc_score(y_true, y_score)
    ci_lo, ci_hi = bootstrap_auc_ci(y_true, y_score, n_bootstrap=N_BOOTSTRAP, seed=SEED)
    g_thresh, g_sens, g_spec = clinical_threshold(y_true, y_score, SPEC_CONSTRAINT)

    print(f"\n{'═'*54}")
    print(f" GLOBAL OOF RESULTS ({len(global_oof)} recordings, {int(y_true.sum())} pos)")
    print(f"   AUC:         {global_auc:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]  95% CI")
    print(f"   Sensitivity: {g_sens:.4f}")
    print(f"   Specificity: {g_spec:.4f}")
    print(f"   Threshold:   {g_thresh:.4f}")
    print(f"   Folds:  mean={np.mean(fold_aucs):.4f}  std={np.std(fold_aucs):.4f}")
    print(f"{'═'*54}")

    # G4 gates
    mean_auc = np.mean(fold_aucs)
    std_auc  = np.std(fold_aucs)
    print(f"[G4a] {'PASS' if mean_auc >= 0.70 else 'FAIL'} — mean fold AUC = {mean_auc:.4f} (threshold 0.70)")
    print(f"[G4b] {'PASS' if std_auc < 0.10 else 'FAIL'}  — std  fold AUC = {std_auc:.4f}  (threshold 0.10)")

    # Save per-fold summary
    pd.DataFrame(fold_rows).to_csv(BASE_RESULTS / "per_fold_summary.csv", index=False)

    # Save final CV report
    pd.DataFrame([{
        "n_folds": N_FOLDS, "n_recordings": len(global_oof),
        "global_auc": global_auc, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "mean_fold_auc": mean_auc, "std_fold_auc": std_auc,
        "global_threshold": g_thresh, "global_sensitivity": g_sens, "global_specificity": g_spec,
        "spec_constraint": SPEC_CONSTRAINT,
    }]).to_csv(BASE_RESULTS / "final_cv_report_v3.csv", index=False)
    print("[BOOTSTRAP_CI] Saved final_cv_report_v3.csv")

    # Comparison table vs prior results
    comparison = pd.DataFrame([
        {"method": "Paper (benchmark)",              "n": 55,  "auc": 0.826, "ci_lo": None,  "ci_hi": None,  "notes": "original paper"},
        {"method": "Baseline Stage2 LR (test-55)",   "n": 55,  "auc": 0.812, "ci_lo": 0.630, "ci_hi": 0.953, "notes": "results/final_report.md"},
        {"method": "Best post-hoc (AT=0.40, Youden)","n": 55,  "auc": 0.839, "ci_lo": None,  "ci_hi": None,  "notes": "results/final_report.md"},
        {"method": "E2E CV v3 (552, 5-fold OOF)",    "n": 552, "auc": global_auc, "ci_lo": ci_lo, "ci_hi": ci_hi, "notes": "this run"},
    ])
    comparison.to_csv(BASE_RESULTS / "comparison_table_v3.csv", index=False)
    print("[BOOTSTRAP_CI] Saved comparison_table_v3.csv")
    print("\n", comparison[["method", "n", "auc", "ci_lo", "ci_hi"]].to_string(index=False))

In [ ]:
# ─── Cell REPRO_TRACK (Optional) ─────────────────────────────────────────────
# Trains on canonical 441/56/55 split to produce a directly comparable test-set number.
# Skipped automatically if < 40 min remain in the session.
import gc, traceback, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score

_remaining = _session_time_remaining()
if _remaining < 2400:
    print(f"[REPRO_TRACK] Skipping — only {_remaining:.0f}s remaining (need ≥ 2400s / 40 min)")
else:
    print(f"[REPRO_TRACK] Starting ({_remaining:.0f}s remaining) ...")
    repro_ckpt_dir = BASE_CKPT / "repro_track" / "finetune"
    repro_ckpt_dir.mkdir(parents=True, exist_ok=True)

    try:
        finetune_train(
            config_path         = CONFIG_PATH,
            device_str          = DEVICE,
            max_batches         = 0,
            processed_root      = str(PROCESSED_ROOT),
            train_csv           = str(SPLITS_DIR / "train.csv"),
            val_csv             = str(SPLITS_DIR / "val.csv"),
            pretrain_checkpoint = str(SHARED_PRETRAIN_CKPT),
            checkpoint_dir      = str(repro_ckpt_dir),
            log_path            = str(BASE_LOGS / "repro_track_loss.csv"),
            quiet               = True,
            save_epoch_ckpts    = False,  # save disk
            config_overrides    = CONFIG_A_OVERRIDES,
            val_every_n_epochs  = VAL_EVERY_N_EPOCHS,
        )

        best_ckpt = repro_ckpt_dir / "best_finetune.pt"
        if best_ckpt.exists():
            model_r = _load_best_model(str(best_ckpt), DEVICE)
            best_at_r, _, _ = at_sweep(
                model_r, SPLITS_DIR / "val.csv", PROCESSED_ROOT,
                train_csv=SPLITS_DIR / "train.csv", device=DEVICE,
                inference_stride=24, n_features=N_FEATURES,
            )
            X_tr_r, y_tr_r, _ = extract_features_for_split(model_r, SPLITS_DIR / "train.csv", PROCESSED_ROOT, best_at_r, 24, DEVICE, N_FEATURES)
            X_vl_r, y_vl_r, _ = extract_features_for_split(model_r, SPLITS_DIR / "val.csv",   PROCESSED_ROOT, best_at_r, 24, DEVICE, N_FEATURES)
            X_te_r, y_te_r, _ = extract_features_for_split(model_r, SPLITS_DIR / "test.csv",  PROCESSED_ROOT, best_at_r, 24, DEVICE, N_FEATURES)
            X_tv_r = np.concatenate([X_tr_r, X_vl_r])
            y_tv_r = np.concatenate([y_tr_r, y_vl_r])
            sc_r, pca_r, lr_r = fit_lr_model(X_tv_r, y_tv_r, C=0.1, use_pca=True)
            test_sc_r  = predict_lr(X_te_r, sc_r, pca_r, lr_r)
            val_sc_r   = predict_lr(X_vl_r, sc_r, pca_r, lr_r)
            repro_auc  = roc_auc_score(y_te_r, test_sc_r)
            r_thresh, r_sens, r_spec = clinical_threshold(y_vl_r, val_sc_r, SPEC_CONSTRAINT)
            r_lo, r_hi = bootstrap_auc_ci(y_te_r, test_sc_r, N_BOOTSTRAP, SEED)

            print(f"\n[REPRO_TRACK] Canonical test (n=55):")
            print(f"   AUC:         {repro_auc:.4f}  [{r_lo:.4f}, {r_hi:.4f}]")
            print(f"   Sensitivity: {r_sens:.4f}  Specificity: {r_spec:.4f}")
            print(f"   Threshold:   {r_thresh:.4f}  Best AT: {best_at_r:.2f}")
            print(f"   Prior best:  AUC=0.839, Sens=0.818, Spec=0.773  (AT=0.40, thr=0.284)")

            pd.DataFrame([{"auc": repro_auc, "ci_lo": r_lo, "ci_hi": r_hi,
                            "sensitivity": r_sens, "specificity": r_spec,
                            "threshold": r_thresh, "best_at": best_at_r, "prior_auc": 0.839}]
            ).to_csv(BASE_RESULTS / "repro_comparison_v3.csv", index=False)
            print("[REPRO_TRACK] Saved repro_comparison_v3.csv")

            # ── Update comparison_table_v3.csv with RepRo row (spec §12.5) ──────
            cmp_path = BASE_RESULTS / "comparison_table_v3.csv"
            if cmp_path.exists():
                try:
                    _cmp = pd.read_csv(cmp_path)
                    _repro_row = pd.DataFrame([{
                        "method": "RepRo Track v3 (441/56/55)",
                        "n":      55,
                        "auc":    repro_auc,
                        "ci_lo":  r_lo,
                        "ci_hi":  r_hi,
                        "notes":  "canonical split repro — AT locked on val",
                    }])
                    # Insert RepRo row before the last row (E2E CV v3) per spec §12.5 table order
                    _cmp = pd.concat([_cmp.iloc[:-1], _repro_row, _cmp.iloc[-1:]], ignore_index=True)
                    _cmp.to_csv(cmp_path, index=False)
                    print("[REPRO_TRACK] Updated comparison_table_v3.csv with RepRo row")
                    print(_cmp[["method", "n", "auc", "ci_lo", "ci_hi"]].to_string(index=False))
                except Exception as _e:
                    print(f"[REPRO_TRACK] Warning: could not update comparison table: {_e}")
            else:
                print("[REPRO_TRACK] Warning: comparison_table_v3.csv not found — run BOOTSTRAP_CI first")

            del model_r
            gc.collect(); torch.cuda.empty_cache()
    except Exception:
        print("[REPRO_TRACK] Failed:")
        traceback.print_exc()


In [ ]:
# ─── Cell EXPORT — Push final results to GitHub ──────────────────────────────
import datetime, zipfile, traceback, subprocess
from pathlib import Path

ROOT_P = Path(REPO_DIR)

def _sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(f"  [sh] {r.stderr[:300]}")
    return r.stdout.strip()

# ── Prerequisites ─────────────────────────────────────────────────────────────
missing_files = []
for k in range(N_FOLDS):
    p = BASE_RESULTS / f"fold{k}_oof_scores.csv"
    if not p.exists():
        missing_files.append(f"fold{k}_oof_scores.csv")
if not (BASE_RESULTS / "final_cv_report_v3.csv").exists():
    missing_files.append("final_cv_report_v3.csv")
if missing_files:
    print(f"[EXPORT] WARNING: Missing files — exporting partial results: {missing_files}")
else:
    print("[EXPORT] All prerequisite files present — proceeding")

# ── Retrieve GitHub token ─────────────────────────────────────────────────────
GITHUB_TOKEN = None
try:
    from google.colab import userdata as _ud
    GITHUB_TOKEN = _ud.get('GITHUB_TOKEN')
    assert GITHUB_TOKEN, "empty token"
    print("[EXPORT] GitHub token retrieved from Colab secrets")
except Exception as e:
    print(f"[EXPORT] Token not found: {e} — push may fail, files downloadable manually")

# ── Zip best checkpoints ──────────────────────────────────────────────────────
zip_path = BASE_CKPT / "best_models_v3.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for k in range(N_FOLDS):
        src = BASE_CKPT / f"fold{k}" / "finetune" / "best_finetune.pt"
        if src.exists():
            zf.write(src, f"fold{k}_best_finetune.pt")
            print(f"  [zip] Added fold{k}_best_finetune.pt")
        else:
            print(f"  [zip] fold{k} best checkpoint missing — skipped")
print(f"[EXPORT] Checkpoint zip: {zip_path}")

# ── Git push ──────────────────────────────────────────────────────────────────
try:
    _sh('git config --global user.email "colab-auto@sentinelfatal2"')
    _sh('git config --global user.name "Colab TrainingAgent"')
    if GITHUB_TOKEN:
        _sh(f'git -C {REPO_DIR} remote set-url origin '
            f'"https://{GITHUB_TOKEN}@github.com/ArielShamay/SentinelFatal2.git"')
    _sh(f'git -C {REPO_DIR} add "{BASE_RESULTS}/" "{BASE_LOGS}/"')
    _sh(f'git -C {REPO_DIR} add -f "{zip_path}"')
    date_str = datetime.date.today().isoformat()
    _sh(f'git -C {REPO_DIR} commit -m "[auto] E2E CV v3 final results — {date_str}"')
    out = _sh(f'git -C {REPO_DIR} push origin master')
    print(f"[EXPORT] Push {'complete' if not out else 'output: ' + out}")
except Exception as e:
    print(f"[EXPORT] Git push error: {e}")
    traceback.print_exc()
    print(f"\n[EXPORT] FALLBACK — download manually from Colab file browser:")
    print(f"   {BASE_RESULTS}/")
    print(f"   {zip_path}")
finally:
    # Always strip token from remote URL
    _sh(f'git -C {REPO_DIR} remote set-url origin "https://github.com/ArielShamay/SentinelFatal2.git"')

print("[EXPORT] Done")